# R27 Agentic Precision Arms - identity FALSE-MERGE removal on the SAME_AS surface

Executor wave for the re-scoped R27 round. H288 killed the recall motivation (merge-over-soft-link delta 0.0 pts);
the only value escalation can buy is **false-merge removal** on the SAME_AS surface at **zero true-merge loss**.

Arms: **H282** single-shot control, **H283** tool-agent (bounded loop), **H284** effort law (K sweep),
**H285** contrarian (no-tools multi-round + fetch-then-judge), **H289** harness tax (Strands vs bare loop).

Discipline: neo4j2 (`bolt://172.19.0.9:7687`) READ-ONLY for evidence; local vLLM `gpt-oss-120b` at `localhost:8010`,
temp 0, <=4 in flight. Gold labels frozen BLIND before any LLM call. Metric everywhere: false-merge detection %,
true-merge preservation %, tokens/pair.

## Imports

In [1]:
import os, re, json, time, hashlib, pickle, unicodedata, collections, statistics, random
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from neo4j import GraphDatabase
from openai import OpenAI
from rapidfuzz import fuzz
from rich.console import Console
from rich.table import Table
con = Console()
con.print("imports ready")

imports ready

## Configuration

In [2]:
NEO4J_URI  = "bolt://172.19.0.9:7687"          # READ-ONLY reference graph (neo4j2)
NEO4J_AUTH = ("neo4j", "kgfoundry")
LLM_BASE   = "http://localhost:8010/v1"
LLM_MODEL  = "gpt-oss-120b"
TEMP       = 0.0
REASONING_EFFORT = "medium"                     # controlled constant across ALL arms - see harness note
MAX_INFLIGHT = 4                                # shared GPU - keep <=4 concurrent
UTC        = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

ROOT   = Path("..").resolve()
H101_PATH   = ROOT/"reports/identity-benchmark-h101-20260707-094448.json"
CHUNK_PKL   = ROOT/"data/interim/h119_chunks.pkl"
CKPT_DIR    = ROOT/"results/r27_precision"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH  = CKPT_DIR/"llm_cache.json"
LABELS_PATH = ROOT/f"reports/r27-precision-labels-frozen-{UTC}.json"
REPORT_PATH = ROOT/f"reports/agentic-precision-arms-r27-{UTC}.json"

t = Table(title="R27 precision-arms config", show_header=False)
for k,v in [("neo4j2 (RO)",NEO4J_URI),("LLM",f"{LLM_MODEL} @ {LLM_BASE}"),
            ("temp",TEMP),("max in-flight",MAX_INFLIGHT),("UTC",UTC),
            ("labels",LABELS_PATH.name),("report",REPORT_PATH.name)]:
    t.add_row(str(k),str(v))
con.print(t)

                      R27 precision-arms config                      
┌───────────────┬───────────────────────────────────────────────────┐
│ neo4j2 (RO)   │ bolt://172.19.0.9:7687                            │
│ LLM           │ gpt-oss-120b @ http://localhost:8010/v1           │
│ temp          │ 0.0                                               │
│ max in-flight │ 4                                                 │
│ UTC           │ 20260708T095229Z                                  │
│ labels        │ r27-precision-labels-frozen-20260708T095229Z.json │
│ report        │ agentic-precision-arms-r27-20260708T095229Z.json  │
└───────────────┴───────────────────────────────────────────────────┘

## Load reference graph (read-only)

In [3]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
def q(cypher, **p):
    with driver.session() as s:
        return [r.data() for r in s.run(cypher, **p)]

CODE_KEYS = ("prop_part_number","prop_us_part_number","prop_canadian_part_number","prop_hcpcs_code",
             "prop_device_code","prop_order_number","prop_model_number_standard",
             "prop_model_number_with_humidifier","prop_model_number_with_heated_tube")
nodes = {}
for r in q("MATCH (e) WHERE e.name IS NOT NULL "
           "RETURN e.id AS id, e.name AS name, labels(e) AS labs, e.description AS desc, "
           "e.source_chunks AS chunks, e.source_documents AS docs, properties(e) AS props"):
    props = r["props"]
    codes = {k: str(props[k]) for k in CODE_KEYS if props.get(k) is not None}
    allprops = {k: props[k] for k in props if k.startswith("prop_") and props[k] is not None}
    labs = [l for l in r["labs"] if l != "Entity"]
    nodes[r["id"]] = dict(name=r["name"], labs=labs, prim=(labs[0] if labs else "Entity"),
                          desc=r["desc"] or "", chunks=r["chunks"] or [], docs=r["docs"] or [],
                          codes=codes, props=allprops)

CONTENT_EXCLUDE = {"MENTIONED_IN","ABOUT","SIMILAR_TO","SAME_AS","HAD_VERSION"}
adj = collections.defaultdict(list)   # id -> [(rel_type, neighbor_id)]
for r in q("MATCH (a)-[x]->(b) WHERE a.id IS NOT NULL AND b.id IS NOT NULL "
           "RETURN a.id AS a, type(x) AS t, b.id AS b"):
    if r["t"] not in CONTENT_EXCLUDE:
        adj[r["a"]].append((r["t"], r["b"])); adj[r["b"]].append((r["t"], r["a"]))

chunk_text = {r["id"]: r["text"] for r in q("MATCH (c:Chunk) RETURN c.id AS id, c.text AS text")}
try:
    pk = pickle.load(open(CHUNK_PKL,"rb"))
    for doc,chs in pk.items():
        for c in chs: chunk_text.setdefault(c["id"], c["text"])
except Exception as e:
    con.print(f"[yellow]chunk pickle skipped: {e}[/yellow]")

con.print(f"named nodes: [cyan]{len(nodes)}[/cyan]  content-adj entities: [cyan]{len(adj)}[/cyan]  "
          f"chunk texts: [cyan]{len(chunk_text)}[/cyan]")

named nodes: 2825  content-adj entities: 2679  chunk texts: 132

## Enumerate the SAME_AS surface and classify (H288 precision-residue classes)

In [4]:
sa = q("MATCH (a)-[:SAME_AS]->(b) RETURN a.id AS aid, b.id AS bid")
code_re = re.compile(r"\b([A-Z]{1,4}\d{1,4}[A-Z]?)\b")
def codes_in(name): return set(m.group(1) for m in code_re.finditer((name or "").upper()))

pairs = []
for e in sa:
    a, b = e["aid"], e["bid"]
    if a not in nodes or b not in nodes: continue
    na, nb = nodes[a], nodes[b]
    shared = codes_in(na["name"]) & codes_in(nb["name"])
    tc = na["prim"] != nb["prim"]
    cs = bool(shared)
    cls = "type_conflict" if tc else ("code_shared" if cs else "clean")
    if tc and cs: cls = "type_conflict+code_shared"
    pairs.append(dict(key=f"{a}|{b}", a=a, b=b, an=na["name"], bn=nb["name"],
                      ta=na["prim"], tb=nb["prim"], shared_code=sorted(shared),
                      type_conflict=tc, code_shared=cs, cls=cls))
con.print(f"SAME_AS pairs: [cyan]{len(pairs)}[/cyan]  "
          f"type_conflict={sum(p['type_conflict'] for p in pairs)}  "
          f"code_shared={sum(p['code_shared'] for p in pairs)}  "
          f"clean={sum(p['cls']=='clean' for p in pairs)}")

SAME_AS pairs: 127  type_conflict=61  code_shared=64  clean=32

## Freeze GOLD labels - BLIND adjudication (before any LLM call)

Each SAME_AS pair is adjudicated by the executor from graph evidence (names, types, descriptions, codes)
applying a fixed rubric, hand-curated over the full enumeration. `MERGE` = the two nodes denote the SAME
real-world entity (a true merge to preserve); `DISTINCT` = a false merge to remove. Where H101 already labels
a pair (`known_false`, `p10_class`), its label wins. Labels are frozen to disk before the LLM harness loads.

In [5]:
# Executor's blind adjudication of the SAME_AS surface (name-pair -> gold), curated from the full
# evidence dump. MERGE = same real-world entity (true merge); everything else is a false merge (DISTINCT).
GOLD_MERGE_NAMEPAIRS = {frozenset(x) for x in [
    ("Alice LoFlo C5 Sidestream module","LoFlo C5 Sidestream module"),
    ("CT2 adult Alice 5","CT2, adult"),
    ("HC230 Product Range","HC230-Series"),
    ("M10 oxygen concentrator","Millennium M10"),
    ("MD300W314B4","Wrist Pulse Oximeter MD300W314B4"),
    ("Trilogy100","Trilogy100 ventilator"),
    ("APAP","Auto"),
    ("SleepStyle 200 Series","HC230-Series"),      # F&P SleepStyle 200 carries model HC230
    ("Air Filter Cover","Air filter cover"),
    ("Apnea Hypopnea Index","Apnea-Hypopnea Index"),
    ("Auto EPAP","Auto-EPAP"),
    ("Auto Off","Auto-Off"),("Auto On","Auto-On"),("Auto Ramp","AutoRamp"),
    ("Auto Start","autoSTART"),
    ("CISPR 11","CISPR11"),
    ("Flow Meter","Flowmeter"),
    ("Full face mask","Full-Face Mask"),
    ("ISO 80601-2-70:2015","ISO 80601-2-70:2015"),
    ("Nasal Cannulas Adult","Nasal cannulas (adult)"),
    ("Performance Tubing (22mm)","Performance Tubing 22mm"),
    ("Pre-heat","Preheat"),
    ("Pressure Start Stop Button","Pressure Start/Stop Button"),
    ("ResMed AirSense 10","ResMed Airsense10"),
    ("Smart Ramp","SmartRamp"),
    ("Ultra-Fine Filter Disposable 1 Pack","Ultra-fine Filter Disposable 1-pack"),
    ("ezRIP module (abdomen)","ezRIP module, abdomen"),
    ("AirFit N20","AirFit N20 Classic"),
    ("AirFit N20","ResMed AirFit N20 Classic"),
]}
# medium-confidence MERGE calls flagged for honest reporting
GOLD_MERGE_LOWCONF = {frozenset(x) for x in [
    ("HC230 Product Range","HC230-Series"),
    ("AirFit N20","AirFit N20 Classic"),
    ("AirFit N20","ResMed AirFit N20 Classic"),
    ("CT2 adult Alice 5","CT2, adult"),
]}

# H101 hard labels win where present
h101 = json.load(open(H101_PATH))
h101_no = set()
for p in h101["pairs"]:
    if p.get("labeled_false"): h101_no.add(frozenset((p["a"], p["b"])))
for p in h101.get("p10_class", []): h101_no.add(frozenset((p["a"], p["b"])))

for p in pairs:
    npair = frozenset((p["an"], p["bn"]))
    if npair in h101_no:
        p["gold"] = "DISTINCT"; p["gold_src"] = "H101"
    elif npair in GOLD_MERGE_NAMEPAIRS:
        p["gold"] = "MERGE"; p["gold_src"] = "executor"
    else:
        p["gold"] = "DISTINCT"; p["gold_src"] = "executor"
    p["gold_lowconf"] = npair in GOLD_MERGE_LOWCONF

ng = collections.Counter(p["gold"] for p in pairs)
con.print(f"frozen gold: [green]{ng['MERGE']} MERGE[/green] (true merges) / "
          f"[red]{ng['DISTINCT']} DISTINCT[/red] (false merges)   total {len(pairs)}")
ct = collections.defaultdict(lambda: collections.Counter())
for p in pairs: ct[p["cls"]][p["gold"]] += 1
tb = Table(title="gold x class"); tb.add_column("class"); tb.add_column("MERGE"); tb.add_column("DISTINCT")
for c,cc in sorted(ct.items()): tb.add_row(c, str(cc["MERGE"]), str(cc["DISTINCT"]))
con.print(tb)

frozen gold: 26 MERGE (true merges) / 101 DISTINCT (false merges)   total 127

                  gold x class                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ class                     ┃ MERGE ┃ DISTINCT ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ clean                     │ 13    │ 19       │
│ code_shared               │ 5     │ 29       │
│ type_conflict             │ 7     │ 24       │
│ type_conflict+code_shared │ 1     │ 29       │
└───────────────────────────┴───────┴──────────┘

In [6]:
# freeze to disk BEFORE any LLM call
frozen = dict(round="R27", utc=UTC, source="SAME_AS surface on neo4j2 (127 pairs) + H101 hard labels",
              metric="false-merge removal at zero true-merge loss",
              n_pairs=len(pairs), n_merge=ng["MERGE"], n_distinct=ng["DISTINCT"],
              pairs=[{k:p[k] for k in ("key","an","bn","ta","tb","cls","shared_code",
                       "type_conflict","code_shared","gold","gold_src","gold_lowconf")} for p in pairs])
json.dump(frozen, open(LABELS_PATH,"w"), indent=1)
con.print(f"[bold green]labels frozen -> {LABELS_PATH.name}[/bold green] ({len(pairs)} pairs, "
          f"{ng['MERGE']} true / {ng['DISTINCT']} false)")

labels frozen -> r27-precision-labels-frozen-20260708T095229Z.json (127 pairs, 26 true / 101 false)

## LLM harness, disk cache, deterministic tools

In [7]:
client = OpenAI(base_url=LLM_BASE, api_key="x")
_cache = json.load(open(CACHE_PATH)) if CACHE_PATH.exists() else {}
_cache_dirty = [0]
def _save_cache():
    json.dump(_cache, open(CACHE_PATH,"w")); _cache_dirty[0] = 0

def llm(messages, max_tokens=1024, tag=""):
    """Cached temp-0 chat completion. Returns (content, total_tokens)."""
    key = hashlib.sha1((tag+"|"+json.dumps(messages)+f"|{max_tokens}|{REASONING_EFFORT}").encode()).hexdigest()
    if key in _cache:
        c = _cache[key]; return c["content"], c["tokens"]
    for attempt in range(4):
        try:
            r = client.chat.completions.create(model=LLM_MODEL, temperature=TEMP,
                                               messages=messages, max_tokens=max_tokens,
                                               extra_body={"reasoning_effort": REASONING_EFFORT})
            content = r.choices[0].message.content or ""
            tok = r.usage.total_tokens
            _cache[key] = dict(content=content, tokens=tok)
            _cache_dirty[0] += 1
            if _cache_dirty[0] >= 25: _save_cache()
            return content, tok
        except Exception as e:
            if attempt == 3: raise
            time.sleep(2*(attempt+1))

def run_pool(items, fn):
    """Map fn over items with <=MAX_INFLIGHT workers, preserving order."""
    out = [None]*len(items)
    with ThreadPoolExecutor(max_workers=MAX_INFLIGHT) as ex:
        futs = {ex.submit(fn, it): i for i, it in enumerate(items)}
        for f in as_completed(futs):
            out[futs[f]] = f.result()
    _save_cache()
    return out

def parse_verdict(text):
    if not text: return None
    m = re.search(r'"?verdict"?\s*[:=]\s*"?(MERGE|DISTINCT|SAME|SAME_AS|YES|NO|DIFFERENT)"?', text, re.I)
    if m:
        v = m.group(1).upper()
        return "MERGE" if v in ("MERGE","SAME","SAME_AS","YES") else "DISTINCT"
    up = text.upper()
    if "DISTINCT" in up or "DIFFERENT" in up: return "DISTINCT"
    if "MERGE" in up or "SAME ENTITY" in up: return "MERGE"
    return None

In [8]:
# ---- deterministic value comparator (H194 gnorm + H208 unit-aware) ----
_TM = dict.fromkeys(map(ord, "\u00ae\u2122\u00a9"), None)
def gnorm(s):
    s = (s or "").translate(_TM); s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00a0"," ").replace("\u00d7","x").replace("*","x").replace("\u00b7","x")
    s = re.sub(r"(?<=\d),(?=\d)", "", s)
    return re.sub(r"\s+", " ", s.casefold()).strip()
XUNIT = {"g":("mass",1.0),"kg":("mass",1000.0),"oz":("mass",28.349523125),
    "lb":("mass",453.59237),"lbs":("mass",453.59237),
    "mm":("len",1.0),"cm":("len",10.0),"inch":("len",25.4),"inches":("len",25.4),"in":("len",25.4),
    "cmh2o":("press",1.0),"hpa":("press",1.0197162),"mbar":("press",1.0197162),"kpa":("press",10.197162),
    "l/min":("flow",1.0),"lpm":("flow",1.0),"ml/min":("flow",0.001)}
_XALT = r"(cmh2o|cm h2o|ml/min|l/min|mm|cm|kg|lbs|lb|pounds|pound|oz|inches|inch|hpa|mbar|kpa|lpm|in|g)"
def quants(text):
    Q=set(); t=gnorm(text)
    for m in re.finditer(r"(\d+(?:\.\d+)?)\s*"+_XALT+r"\b", t):
        u=m.group(2).replace(" ","")
        if u in XUNIT:
            dim,f=XUNIT[u]; Q.add((round(float(m.group(1))*f,3), dim))
    return Q

def value_comparator(a, b):
    na, nb = nodes[a], nodes[b]
    conflicts=[]; matches=[]
    for k in set(na["props"]) & set(nb["props"]):
        va, vb = gnorm(str(na["props"][k])), gnorm(str(nb["props"][k]))
        if not va or not vb: continue
        (matches if va==vb else conflicts).append((k, na["props"][k], nb["props"][k]))
    qa = quants(na["name"]+" "+na["desc"]); qb = quants(nb["name"]+" "+nb["desc"])
    qconf = [ (x,y) for x in qa for y in qb if x[1]==y[1] and x[0]!=y[0] ]
    shared_code = sorted(set(m.group(1) for m in code_re.finditer((na["name"]).upper()))
                       & set(m.group(1) for m in code_re.finditer((nb["name"]).upper())))
    return dict(prop_conflicts=conflicts[:6], prop_matches=matches[:6],
                quantity_conflict=bool(qconf), shared_name_codes=shared_code,
                verdict=("CONFLICT" if conflicts or qconf else ("MATCH" if matches else "NO_SHARED_KEYS")))

# ---- evidence fetch tools (read-only, deterministic) ----
def tool_neighborhood(a, maxn=12):
    out=[]
    for rel, nb in adj.get(a, [])[:maxn*2]:
        if nb in nodes: out.append(f"-{rel}-> [{nodes[nb]['prim']}] {nodes[nb]['name']}")
        if len(out)>=maxn: break
    return out
def tool_source_span(a, maxchars=700):
    txt=" ".join(chunk_text.get(c,"") for c in nodes[a]["chunks"][:3])
    return txt[:maxchars] if txt.strip() else "(no source chunk text available)"
def tool_string_code(a, b):
    na,nb=nodes[a],nodes[b]
    return dict(name_sim=fuzz.token_set_ratio(na["name"].lower(), nb["name"].lower()),
                codes_a=sorted(codes_in(na["name"])), codes_b=sorted(codes_in(nb["name"])),
                shared_codes=sorted(codes_in(na["name"]) & codes_in(nb["name"])))
con.print("harness, comparator and 4 tools ready: neighborhood / source_span / value_comparator / string_code")

harness, comparator and 4 tools ready: neighborhood / source_span / value_comparator / string_code

In [9]:
# ---- context builders + prompts ----
SYS_JUDGE = ("You are an entity-resolution adjudicator for a medical-device knowledge graph. "
    "Decide whether two graph nodes denote the SAME real-world entity (MERGE) or DIFFERENT entities "
    "(DISTINCT). A device is DISTINCT from its accessory; a mask is DISTINCT from a battery even if they "
    "share a code token; sibling model variants are DISTINCT; pure spelling/spacing/case aliases are MERGE. "
    'Reply with ONLY a JSON object: {"verdict":"MERGE"|"DISTINCT","confidence":0-1,"rationale":"<=20 words"}.')

def base_ctx(p):
    a,b=p["a"],p["b"]; na,nb=nodes[a],nodes[b]
    def nb1(x): return "; ".join(tool_neighborhood(x,6)) or "(none)"
    return (f'A: [{na["prim"]}] "{na["name"]}"\n  desc: {na["desc"][:200] or "(none)"}\n'
            f'  neighbors: {nb1(a)}\n'
            f'B: [{nb["prim"]}] "{nb["name"]}"\n  desc: {nb["desc"][:200] or "(none)"}\n'
            f'  neighbors: {nb1(b)}\n'
            f'shared source docs: {sorted(set(na["docs"]) & set(nb["docs"])) or "(none)"}')

def single_shot(p, max_tokens=1024, tag="H282"):
    msgs=[{"role":"system","content":SYS_JUDGE},
          {"role":"user","content":"Judge this pair.\n\n"+base_ctx(p)}]
    content, tok = llm(msgs, max_tokens=max_tokens, tag=tag)
    return dict(verdict=parse_verdict(content), tokens=tok, raw=content[:200])
con.print("context builder + single-shot judge ready")

context builder + single-shot judge ready

## H282 - the single-shot control

One call per pair, standard context (names, descriptions, types, 1-hop neighborhoods, shared provenance).
Three repeats on a 20-pair subsample for variance honesty.

In [10]:
def metrics(pairs_list, verdicts):
    fm = [(p,v) for p,v in zip(pairs_list,verdicts) if p["gold"]=="DISTINCT"]
    tm = [(p,v) for p,v in zip(pairs_list,verdicts) if p["gold"]=="MERGE"]
    fm_det = sum(v=="DISTINCT" for _,v in fm)/len(fm) if fm else None
    tm_pres= sum(v=="MERGE" for _,v in tm)/len(tm) if tm else None
    losses = [p for p,v in tm if v=="DISTINCT"]
    unresolved = sum(v is None for v in verdicts)
    return dict(n=len(pairs_list), n_false=len(fm), n_true=len(tm),
                false_merge_detection=fm_det, true_merge_preservation=tm_pres,
                true_merge_losses=[(p["an"],p["bn"],p["gold_lowconf"]) for p in losses],
                n_true_merge_loss=len(losses), unresolved=unresolved)
def per_class(pairs_list, verdicts):
    out={}
    for cls in sorted(set(p["cls"] for p in pairs_list)):
        sub=[(p,v) for p,v in zip(pairs_list,verdicts) if p["cls"]==cls]
        out[cls]=metrics([p for p,_ in sub],[v for _,v in sub])
    return out

res_h282 = run_pool(pairs, lambda p: single_shot(p, tag="H282"))
v282 = [r["verdict"] for r in res_h282]
tok282 = [r["tokens"] for r in res_h282]
M282 = metrics(pairs, v282); M282["tokens_per_pair"]=round(statistics.mean(tok282),1)
con.print(f"[bold]H282[/bold] false-merge detection {M282['false_merge_detection']:.1%}  "
          f"true-merge preservation {M282['true_merge_preservation']:.1%}  "
          f"tokens/pair {M282['tokens_per_pair']:.0f}  losses={M282['n_true_merge_loss']}")
if M282["true_merge_losses"]:
    con.print("[red]H282 true-merge LOSSES:[/red]", M282["true_merge_losses"])
json.dump({"per_class":per_class(pairs,v282),**M282}, open(CKPT_DIR/"h282.json","w"), indent=1)

H282 false-merge detection 98.0%  true-merge preservation 69.2%  tokens/pair 513  losses=8

H282 true-merge LOSSES:
[
    ('MD300W314B4', 'Wrist Pulse Oximeter MD300W314B4', False),
    ('APAP', 'Auto', False),
    ('SleepStyle 200 Series', 'HC230-Series', False),
    ('Apnea Hypopnea Index', 'Apnea-Hypopnea Index', False),
    ('Auto EPAP', 'Auto-EPAP', False),
    ('Flow Meter', 'Flowmeter', False),
    ('AirFit N20', 'AirFit N20 Classic', True),
    ('AirFit N20', 'ResMed AirFit N20 Classic', True)
]

In [11]:
# variance: 3 repeats on a 20-pair stratified subsample
sub20 = ([p for p in pairs if p["gold"]=="MERGE"][:7] +
         [p for p in pairs if p["gold"]=="DISTINCT"][:13])
rep_metrics=[]
for rep in range(3):
    vr = run_pool(sub20, lambda p: single_shot(p, tag=f"H282rep{rep}"))
    mm = metrics(sub20,[r["verdict"] for r in vr])
    rep_metrics.append((mm["false_merge_detection"], mm["true_merge_preservation"]))
fmd=[r[0] for r in rep_metrics]; tmp=[r[1] for r in rep_metrics]
con.print(f"H282 3-repeat on 20 subsample: FM-det {[round(x,3) for x in fmd]}  TM-pres {[round(x,3) for x in tmp]}")
H282_var=dict(repeats=rep_metrics, fm_det_range=[min(fmd),max(fmd)], tm_pres_range=[min(tmp),max(tmp)])
con.print(f"[dim]variance FM-det spread {max(fmd)-min(fmd):.1%}, TM-pres spread {max(tmp)-min(tmp):.1%}[/dim]")

H282 3-repeat on 20 subsample: FM-det [0.923, 0.923, 0.923]  TM-pres [0.714, 0.857, 0.714]

variance FM-det spread 0.0%, TM-pres spread 14.3%

## H283 - the tool-agent (bounded loop, K=4)

Closed toolset: graph neighborhood, source-span, deterministic value comparator, string/code analysis.
Plain-python ReAct loop, K=4 cap. Bar (re-aimed): >= 10 pts better false-merge detection than H282
at <= 3x tokens, zero true-merge loss.

In [12]:
# Investigator protocol: the agent decides via TOOLS, not a verdict-only reply. The verdict comes ONLY
# through the answer action. (Harness note: gpt-oss at high reasoning_effort emits empty final-channel
# content under this multi-step protocol; all arms run at reasoning_effort="medium" for a controlled compare.)
SYS_AGENT = (
 "You are an entity-resolution investigator for a medical-device knowledge graph. Decide whether two graph "
 "nodes denote the SAME real-world entity (MERGE) or DIFFERENT entities (DISTINCT). Rules: a device is DISTINCT "
 "from its accessory; a mask is DISTINCT from a battery even if they share a code token; sibling model variants "
 "are DISTINCT; pure spelling/spacing/case aliases are MERGE. You investigate by calling TOOLS to gather evidence "
 "BEFORE deciding. Output ONLY your single next action as one JSON object - never multiple objects, never plan "
 "ahead. Prefer to call at least one tool before answering.\nTOOLS:\n"
 '  {"action":"neighborhood","of":"A"}   - typed 1-hop neighbors of node A (or "B")\n'
 '  {"action":"source_span","of":"A"}    - verbatim source-chunk text for node A (or "B")\n'
 '  {"action":"value_comparator"}         - deterministic structured prop/quantity conflict check\n'
 '  {"action":"string_code"}              - name similarity + shared model-code tokens\n'
 'FINAL (after gathering evidence): {"action":"answer","verdict":"MERGE"|"DISTINCT","rationale":"<=15 words"}')

def exec_tool(p, act):
    a,b=p["a"],p["b"]
    who = b if str(act.get("of","")).upper()=="B" else a
    name = act.get("action")
    if name=="neighborhood": return json.dumps(tool_neighborhood(who))
    if name=="source_span":  return tool_source_span(who)
    if name=="value_comparator": return json.dumps(value_comparator(a,b))
    if name=="string_code":  return json.dumps(tool_string_code(a,b))
    return "unknown tool"

def parse_action(text):
    """Parse the FIRST action JSON object (stepwise execution), tolerant of ``` fences."""
    if not text: return None
    text = re.sub(r'```(?:json)?', '', text)
    for c in re.findall(r'\{[^{}]*"action"[^{}]*\}', text, re.S):
        try: return json.loads(c)
        except Exception:
            try: return json.loads(c.replace("'", '"'))
            except Exception: continue
    return None

def tool_agent(p, K=4, tag="H283"):
    msgs=[{"role":"system","content":SYS_AGENT},
          {"role":"user","content":"Judge this pair. Fetch evidence if useful, then answer.\n\n"+base_ctx(p)}]
    tot=0; calls=[]; fetch_sigs=set(); new_fetch_per_round=[]
    for rnd in range(K):
        content, tok = llm(msgs, max_tokens=1400, tag=f"{tag}r{rnd}"); tot+=tok
        act=parse_action(content)
        if act is None or act.get("action")=="answer":
            return dict(verdict=parse_verdict(content), tokens=tot, rounds=rnd+1, tools=calls,
                        new_fetch_per_round=new_fetch_per_round)
        sig=(act.get("action"), str(act.get("of","")))
        new_fetch_per_round.append(sig not in fetch_sigs); fetch_sigs.add(sig)
        calls.append(act.get("action"))
        obs=exec_tool(p, act)
        msgs.append({"role":"assistant","content":content[:600]})
        msgs.append({"role":"user","content":f"OBSERVATION ({act.get('action')}): {obs[:800]}\n"
                     "Fetch more or answer now."})
    msgs.append({"role":"user","content":'Answer now: {"verdict":"MERGE"|"DISTINCT",...}'})
    content, tok = llm(msgs, max_tokens=600, tag=f"{tag}rF"); tot+=tok
    return dict(verdict=parse_verdict(content), tokens=tot, rounds=K, tools=calls,
                new_fetch_per_round=new_fetch_per_round)

res_h283 = run_pool(pairs, lambda p: tool_agent(p, K=4, tag="H283"))
v283=[r["verdict"] for r in res_h283]; tok283=[r["tokens"] for r in res_h283]
M283 = metrics(pairs, v283); M283["tokens_per_pair"]=round(statistics.mean(tok283),1)
M283["mean_rounds"]=round(statistics.mean(r["rounds"] for r in res_h283),2)
M283["tool_usage"]=dict(collections.Counter(t for r in res_h283 for t in r["tools"]))
delta_fm = (M283["false_merge_detection"]-M282["false_merge_detection"])*100
tok_ratio = M283["tokens_per_pair"]/M282["tokens_per_pair"]
con.print(f"[bold]H283[/bold] FM-det {M283['false_merge_detection']:.1%} (delta {delta_fm:+.1f} pts vs H282)  "
          f"TM-pres {M283['true_merge_preservation']:.1%}  tokens/pair {M283['tokens_per_pair']:.0f} "
          f"({tok_ratio:.1f}x)  losses={M283['n_true_merge_loss']}")
con.print(f"[dim]tool usage: {M283['tool_usage']}  mean rounds {M283['mean_rounds']}[/dim]")
if M283["true_merge_losses"]: con.print("[red]H283 true-merge LOSSES:[/red]", M283["true_merge_losses"])
H283_bar = dict(delta_fm_pts=round(delta_fm,1), token_ratio=round(tok_ratio,2),
                zero_true_loss=(M283["n_true_merge_loss"]==0),
                pass_bar=(delta_fm>=10 and tok_ratio<=3 and M283["n_true_merge_loss"]==0))
json.dump({"per_class":per_class(pairs,v283),**M283,"bar":H283_bar}, open(CKPT_DIR/"h283.json","w"), indent=1)
con.print(f"[bold]{'PASS' if H283_bar['pass_bar'] else 'FAIL'}[/bold] registered bar (>=10pts, <=3x, zero loss)")

H283 FM-det 85.1% (delta -12.9 pts vs H282)  TM-pres 80.8%  tokens/pair 2017 (3.9x)  losses=5

tool usage: {'neighborhood': 56, 'source_span': 118, 'string_code': 18}  mean rounds 2.5

H283 true-merge LOSSES:
[
    ('MD300W314B4', 'Wrist Pulse Oximeter MD300W314B4', False),
    ('SleepStyle 200 Series', 'HC230-Series', False),
    ('Flow Meter', 'Flowmeter', False),
    ('AirFit N20', 'AirFit N20 Classic', True),
    ('AirFit N20', 'ResMed AirFit N20 Classic', True)
]

FAIL registered bar (>=10pts, <=3x, zero loss)

## H284 - the effort law (K sweep on a 30-pair subsample)

Sweep K in {1,2,4,8} on a stratified 30-pair subsample; report accuracy-vs-K and whether rounds past 3
fetch NEW evidence or re-argue.

In [13]:
sub30 = ([p for p in pairs if p["gold"]=="MERGE"][:10] +
         [p for p in pairs if p["gold"]=="DISTINCT"][:20])
h284={}
for K in (1,2,4,8):
    rr = run_pool(sub30, lambda p: tool_agent(p, K=K, tag=f"H284K{K}"))
    vv=[r["verdict"] for r in rr]
    mm=metrics(sub30, vv); mm["tokens_per_pair"]=round(statistics.mean(r["tokens"] for r in rr),1)
    mm["mean_rounds"]=round(statistics.mean(r["rounds"] for r in rr),2)
    late_new = sum(nf for r in rr for nf in r["new_fetch_per_round"][3:])
    late_tot = sum(len(r["new_fetch_per_round"][3:]) for r in rr)
    mm["late_round_new_fetch"]=f"{late_new}/{late_tot}"
    h284[K]=mm
    con.print(f"K={K}: FM-det {mm['false_merge_detection']:.1%} TM-pres {mm['true_merge_preservation']:.1%} "
              f"tok/pair {mm['tokens_per_pair']:.0f} rounds {mm['mean_rounds']} late-new {mm['late_round_new_fetch']}")
base=h284[1]["false_merge_detection"]; top=h284[8]["false_merge_detection"]; lift=top-base
cap4=(h284[4]["false_merge_detection"]-base)
h284_knee=dict(base_K1=base, top_K8=top, lift_pts=round(lift*100,1),
               frac_captured_by_K4=(round(cap4/lift,2) if lift else None),
               climbs_past_K4=(h284[8]["false_merge_detection"]>h284[4]["false_merge_detection"]+1e-9))
con.print(f"[dim]effort knee: lift {h284_knee['lift_pts']} pts, K<=4 captures "
          f"{h284_knee['frac_captured_by_K4']}, climbs past K4: {h284_knee['climbs_past_K4']}[/dim]")
json.dump({str(k):v for k,v in h284.items()}|{"knee":h284_knee}, open(CKPT_DIR/"h284.json","w"), indent=1)

K=1: FM-det 80.0% TM-pres 60.0% tok/pair 1408 rounds 1 late-new 0/0

K=2: FM-det 90.0% TM-pres 80.0% tok/pair 2230 rounds 1.97 late-new 0/0

K=4: FM-det 100.0% TM-pres 60.0% tok/pair 1999 rounds 2.43 late-new 0/0

K=8: FM-det 95.0% TM-pres 60.0% tok/pair 2348 rounds 2.67 late-new 3/4

effort knee: lift 15.0 pts, K<=4 captures 1.33, climbs past K4: False

## H285 - contrarian: deliberation is theater

(i) no-tools multi-round arm (same K, no tools); (ii) fetch-then-judge (deterministically fetch source spans
+ comparator verdicts for every pair, then ONE call). If fetch-then-judge matches the tool-agent within 3 pts,
the shipped form is fetch-then-judge.

In [14]:
SYS_NOTOOL = SYS_JUDGE + ("\nThink step by step across up to 4 short rounds of reasoning, then answer. "
                          "You have NO tools - reason only from the context given.")
def no_tools_agent(p, K=4, tag="H285nt"):
    msgs=[{"role":"system","content":SYS_NOTOOL},
          {"role":"user","content":"Deliberate then answer.\n\n"+base_ctx(p)}]
    tot=0; content=""
    for rnd in range(K):
        content,tok=llm(msgs,max_tokens=900,tag=f"{tag}r{rnd}"); tot+=tok
        v=parse_verdict(content)
        if v and ('"verdict"' in content or rnd==K-1 or "answer" in content.lower()):
            return dict(verdict=v, tokens=tot, rounds=rnd+1)
        msgs.append({"role":"assistant","content":content[:600]})
        msgs.append({"role":"user","content":"Continue reasoning or give final JSON verdict."})
    return dict(verdict=parse_verdict(content), tokens=tot, rounds=K)

def fetch_then_judge(p, tag="H285ftj"):
    a,b=p["a"],p["b"]
    evid=(base_ctx(p)+"\n\nFETCHED EVIDENCE:\n"
          f"value_comparator: {json.dumps(value_comparator(a,b))}\n"
          f"string_code: {json.dumps(tool_string_code(a,b))}\n"
          f"A source span: {tool_source_span(a,400)}\n"
          f"B source span: {tool_source_span(b,400)}")
    content,tok=llm([{"role":"system","content":SYS_JUDGE},
                     {"role":"user","content":"Judge using all evidence.\n\n"+evid}],
                    max_tokens=1024, tag=tag)
    return dict(verdict=parse_verdict(content), tokens=tok)

res_nt = run_pool(pairs, lambda p: no_tools_agent(p, tag="H285nt"))
res_ftj= run_pool(pairs, lambda p: fetch_then_judge(p, tag="H285ftj"))
vnt=[r["verdict"] for r in res_nt]; vftj=[r["verdict"] for r in res_ftj]
Mnt=metrics(pairs,vnt); Mnt["tokens_per_pair"]=round(statistics.mean(r["tokens"] for r in res_nt),1)
Mftj=metrics(pairs,vftj); Mftj["tokens_per_pair"]=round(statistics.mean(r["tokens"] for r in res_ftj),1)
con.print(f"no-tools  FM-det {Mnt['false_merge_detection']:.1%} TM-pres {Mnt['true_merge_preservation']:.1%} "
          f"tok {Mnt['tokens_per_pair']:.0f} losses {Mnt['n_true_merge_loss']}")
con.print(f"fetch-then-judge FM-det {Mftj['false_merge_detection']:.1%} TM-pres {Mftj['true_merge_preservation']:.1%} "
          f"tok {Mftj['tokens_per_pair']:.0f} losses {Mftj['n_true_merge_loss']}")
ftj_vs_agent = (Mftj["false_merge_detection"]-M283["false_merge_detection"])*100
nt_vs_single = (Mnt["false_merge_detection"]-M282["false_merge_detection"])*100
H285=dict(single_shot=M282["false_merge_detection"], no_tools=Mnt["false_merge_detection"],
          tool_agent=M283["false_merge_detection"], fetch_then_judge=Mftj["false_merge_detection"],
          notools_vs_single_pts=round(nt_vs_single,1), ftj_vs_agent_pts=round(ftj_vs_agent,1),
          ftj_matches_agent_within_3=abs(ftj_vs_agent)<=3,
          shipped_form=("fetch-then-judge" if abs(ftj_vs_agent)<=3 else "tool-agent"))
con.print(f"[bold]H285[/bold] no-tools vs single {nt_vs_single:+.1f} pts; "
          f"fetch-then-judge vs agent {ftj_vs_agent:+.1f} pts -> shipped: [green]{H285['shipped_form']}[/green]")
json.dump({"no_tools":{**Mnt,"per_class":per_class(pairs,vnt)},
           "fetch_then_judge":{**Mftj,"per_class":per_class(pairs,vftj)},"three_way":H285},
          open(CKPT_DIR/"h285.json","w"), indent=1)

no-tools  FM-det 99.0% TM-pres 73.1% tok 539 losses 7

fetch-then-judge FM-det 97.0% TM-pres 65.4% tok 816 losses 9

H285 no-tools vs single +1.0 pts; fetch-then-judge vs agent +11.9 pts -> shipped: tool-agent

## H289 - the harness tax (Strands vs bare loop)

Run a 20-pair matched sample through a Strands Agents SDK loop with the same tools/policy; report token
overhead. If Strands cannot run against the local endpoint quickly, record UNTESTABLE-with-reason.

In [15]:
sub20b = sub20
bare = run_pool(sub20b, lambda p: tool_agent(p, K=4, tag="H289bare"))
bare_tok = round(statistics.mean(r["tokens"] for r in bare),1)
try:
    import importlib; importlib.import_module("strands"); strands_ok=True; strands_reason=""
except Exception as e:
    strands_ok=False; strands_reason=f"{type(e).__name__}: {e}"
if strands_ok:
    H289=dict(status="TESTED", note="strands importable", bare_tokens_per_pair=bare_tok)
else:
    H289=dict(status="UNTESTABLE",
              reason=("Strands Agents SDK not installed in the kg-cli .venv and not a declared pyproject "
                      f"dependency ({strands_reason}); per directive, record UNTESTABLE and move on rather "
                      "than burn hours plumbing a framework the round would price out anyway."),
              bare_tokens_per_pair=bare_tok,
              bare_loop_ships="bare loop is the reference harness; H290 prices against it")
con.print(f"[bold]H289[/bold] {H289['status']} - bare-loop {bare_tok:.0f} tok/pair (matched 20)")
con.print(f"[dim]{H289.get('reason','')}[/dim]")
json.dump(H289, open(CKPT_DIR/"h289.json","w"), indent=1)

H289 UNTESTABLE - bare-loop 2449 tok/pair (matched 20)

Strands Agents SDK not installed in the kg-cli .venv and not a declared pyproject dependency (ModuleNotFoundError: 
No module named 'strands'); per directive, record UNTESTABLE and move on rather than burn hours plumbing a 
framework the round would price out anyway.

## H290-ready cost table + assemble report

In [16]:
def cost_row(name, M):
    return dict(arm=name, false_merge_detection=M.get("false_merge_detection"),
                true_merge_preservation=M.get("true_merge_preservation"),
                n_true_merge_loss=M.get("n_true_merge_loss"), tokens_per_pair=M.get("tokens_per_pair"))
cost_table=[cost_row("H282 single-shot",M282), cost_row("H285 no-tools",Mnt),
            cost_row("H285 fetch-then-judge",Mftj), cost_row("H283 tool-agent K=4",M283)]
ct=Table(title="H290 cost frontier (false-merge removal @ zero true loss)")
for c in ["arm","FM-det","TM-pres","true-loss","tok/pair"]: ct.add_column(c)
for r in cost_table:
    ct.add_row(r["arm"], f"{r['false_merge_detection']:.1%}", f"{r['true_merge_preservation']:.1%}",
               str(r["n_true_merge_loss"]), f"{r['tokens_per_pair']:.0f}")
con.print(ct)

report=dict(round="R27", wave="precision-arms", utc=UTC, author="Claude (Opus executor)",
    endpoint=f"{LLM_MODEL}@{LLM_BASE}", temp=TEMP, labels_frozen=str(LABELS_PATH.name),
    eval_set=dict(n=len(pairs), n_true_merge=ng["MERGE"], n_false_merge=ng["DISTINCT"],
        classes=dict(collections.Counter(p["cls"] for p in pairs))),
    H282=dict(**M282, per_class=per_class(pairs,v282), variance=H282_var),
    H283=dict(**M283, per_class=per_class(pairs,v283), bar=H283_bar),
    H284={str(k):v for k,v in h284.items()}|{"knee":h284_knee},
    H285=H285, H289=H289, cost_table=cost_table)
json.dump(report, open(REPORT_PATH,"w"), indent=1)
_save_cache()
con.print(f"[bold green]report -> {REPORT_PATH.name}[/bold green]")
driver.close()

     H290 cost frontier (false-merge removal @ zero true loss)     
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┓
┃ arm                   ┃ FM-det ┃ TM-pres ┃ true-loss ┃ tok/pair ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━┩
│ H282 single-shot      │ 98.0%  │ 69.2%   │ 8         │ 513      │
│ H285 no-tools         │ 99.0%  │ 73.1%   │ 7         │ 539      │
│ H285 fetch-then-judge │ 97.0%  │ 65.4%   │ 9         │ 816      │
│ H283 tool-agent K=4   │ 85.1%  │ 80.8%   │ 5         │ 2017     │
└───────────────────────┴────────┴─────────┴───────────┴──────────┘

report -> agentic-precision-arms-r27-20260708T095229Z.json

## Verdicts

Grounded in the cells' own outputs above. Registered bars applied as re-aimed for the precision re-scope:
false-merge removal at zero true-merge loss.

In [17]:
vt=Table(title="R27 precision-arms verdicts")
for c in ["arm","result"]: vt.add_column(c)
vt.add_row("H282 single-shot", f"control: FM-det {M282['false_merge_detection']:.0%}, "
           f"TM-pres {M282['true_merge_preservation']:.0%}, {M282['tokens_per_pair']:.0f} tok/pair")
vt.add_row("H283 tool-agent", f"{'PASS' if H283_bar['pass_bar'] else 'FAIL'} - delta {H283_bar['delta_fm_pts']:+} pts, "
           f"{H283_bar['token_ratio']}x tokens, {'zero' if H283_bar['zero_true_loss'] else 'NONZERO'} true-loss")
vt.add_row("H284 effort law", f"lift {h284_knee['lift_pts']} pts K1->K8, K<=4 captures "
           f"{h284_knee['frac_captured_by_K4']}, climbs past K4: {h284_knee['climbs_past_K4']}")
vt.add_row("H285 contrarian", f"shipped form: {H285['shipped_form']} "
           f"(ftj vs agent {H285['ftj_vs_agent_pts']:+} pts, no-tools vs single {H285['notools_vs_single_pts']:+} pts)")
vt.add_row("H289 harness tax", H289["status"])
con.print(vt)

                                     R27 precision-arms verdicts                                     
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ arm              ┃ result                                                                         ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ H282 single-shot │ control: FM-det 98%, TM-pres 69%, 513 tok/pair                                 │
│ H283 tool-agent  │ FAIL - delta -12.9 pts, 3.93x tokens, NONZERO true-loss                        │
│ H284 effort law  │ lift 15.0 pts K1->K8, K<=4 captures 1.33, climbs past K4: False                │
│ H285 contrarian  │ shipped form: tool-agent (ftj vs agent +11.9 pts, no-tools vs single +1.0 pts) │
│ H289 harness tax │ UNTESTABLE                                                                     │
└──────────────────┴────────────────────────────────────────────────────────────────────────────────┘